# Tests de integración de OP-10 `write_flows`

Notebook orientado a verificar el comportamiento end-to-end de la operación pública
OP-10 `write_flows`, considerando:

- materialización del bundle formal en disco;
- coherencia de `summary`, `parameters`, evento y sidecar;
- preservación del estado vivo relevante del `FlowDataset`;
- persistencia opcional de `flow_to_trips`;
- comportamiento observable de los backends Feather y Parquet;
- fallas fatales de escritura por colisión de destino.

Este notebook contiene únicamente integration tests de `write_flows`.
No incluye tests de `read_flows`, ni round-trips write/read, ni smoke tests.

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:

- imports generales;
- imports del módulo;
- helpers de testing reutilizables;
- fixtures de `FlowDataset` suficientemente ricas;
- carpeta local para artefactos persistidos;
- y configuración básica de display.

Los artefactos de los tests se escriben bajo una carpeta local
`./tmp_integration_write_flows`, ubicada en la misma raíz del notebook.

### 0.1 Imports generales

Qué prepara: imports base, utilidades de filesystem, PyArrow para inspección de Parquet
y `deepcopy` para no contaminar fixtures entre tests.

In [1]:
import copy
import json
import shutil
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

### 0.2 Imports del módulo

In [2]:
from pylondrina.datasets import FlowDataset
from pylondrina.errors import ExportError
from pylondrina.io.flows import (
    write_flows,
    WriteFlowsOptions,
)

### 0.3 Helpers de testing reutilizables

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def parquet_has_dictionary_encoding(parquet_path: Path, column_name: str) -> bool:
    pf = pq.ParquetFile(parquet_path)
    try:
        names = pf.schema_arrow.names
        idx = names.index(column_name)
        encodings = {
            str(enc).upper()
            for enc in pf.metadata.row_group(0).column(idx).encodings
        }
        return any("DICTIONARY" in enc for enc in encodings)
    finally:
        pf.close()


INTEGRATION_ROOT = Path("./tmp_integration_write_flows")
ARTIFACTS_ROOT = INTEGRATION_ROOT / "artifacts"


def reset_integration_root() -> Path:
    if INTEGRATION_ROOT.exists():
        shutil.rmtree(INTEGRATION_ROOT)
    ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
    return INTEGRATION_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = ARTIFACTS_ROOT / case_name
    if case_dir.exists():
        shutil.rmtree(case_dir)
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir

### 0.4 Fixtures de flows ricas para integración

Qué prepara:

- una tabla de flows con varias columnas analíticas y de segmentación;
- un auxiliar `flow_to_trips` opcional;
- `FlowDataset` realista con `aggregation_spec`, `metadata`, `events`,
  `provenance` y `source_trips` vivo en memoria.

In [4]:
ORIGINS = [
    "8828308281fffff",
    "8828308283fffff",
    "8828308285fffff",
    "8828308287fffff",
]

DESTINATIONS = [
    "8828308291fffff",
    "8828308293fffff",
    "8828308295fffff",
    "8828308297fffff",
    "8828308299fffff",
]

MODES = ["bus", "metro", "car"]
PURPOSES = ["work", "education", "shopping", "leisure"]
DAY_TYPES = ["weekday", "weekend"]
GENDERS = ["female", "male"]
INCOME_Q = ["1", "3", "5"]
TIME_PERIODS = ["morning_peak", "midday", "afternoon_peak"]


def make_rich_flows_df(*, repeat_blocks: int = 1) -> pd.DataFrame:
    rows = []
    base_ts = pd.Timestamp("2026-04-01T06:00:00Z")

    idx = 0
    for rep in range(repeat_blocks):
        for origin in ORIGINS:
            for destination in DESTINATIONS:
                for mode in MODES:
                    for day_type in DAY_TYPES:
                        for gender in GENDERS:
                            purpose = PURPOSES[idx % len(PURPOSES)]
                            income_q = INCOME_Q[idx % len(INCOME_Q)]
                            time_period = TIME_PERIODS[idx % len(TIME_PERIODS)]

                            window_start = (
                                base_ts
                                + pd.Timedelta(hours=(idx % 10))
                                + pd.Timedelta(days=rep)
                            )
                            window_end = window_start + pd.Timedelta(hours=1)

                            flow_count = 5 + (idx % 17)
                            flow_value = round(
                                flow_count
                                * (
                                    1.0
                                    + (
                                        0.15
                                        if mode == "metro"
                                        else 0.05
                                        if mode == "bus"
                                        else 0.25
                                    )
                                ),
                                3,
                            )

                            rows.append(
                                {
                                    "flow_id": f"f_{rep:02d}_{idx:05d}",
                                    "origin_h3_index": origin,
                                    "destination_h3_index": destination,
                                    "flow_count": int(flow_count),
                                    "flow_value": float(flow_value),
                                    "mode": mode,
                                    "purpose": purpose,
                                    "day_type": day_type,
                                    "user_gender": gender,
                                    "income_quintile": income_q,
                                    "time_period": time_period,
                                    "window_start_utc": window_start,
                                    "window_end_utc": window_end,
                                    "avg_trip_weight": round(
                                        0.8 + (idx % 9) * 0.21,
                                        3,
                                    ),
                                    "segment_label": f"{mode}|{day_type}|{gender}",
                                }
                            )
                            idx += 1

    return pd.DataFrame(rows)


def make_flow_to_trips_df(
    flows_df: pd.DataFrame,
    *,
    links_per_flow: int = 3,
) -> pd.DataFrame:
    rows = []
    movement_counter = 0

    for _, row in flows_df.iterrows():
        for _ in range(links_per_flow):
            movement_counter += 1
            rows.append(
                {
                    "flow_id": row["flow_id"],
                    "movement_id": f"m_{movement_counter:07d}",
                }
            )

    return pd.DataFrame(rows)


def make_rich_flowdataset(
    *,
    repeat_blocks: int = 1,
    with_trip_links: bool = False,
    validated: bool = False,
    dataset_id: str = "flow-dset-integration-001",
) -> FlowDataset:
    flows_df = make_rich_flows_df(repeat_blocks=repeat_blocks)
    flow_to_trips_df = (
        make_flow_to_trips_df(flows_df)
        if with_trip_links
        else None
    )

    aggregation_spec = {
        "h3_resolution": 8,
        "group_by": ["mode", "day_type", "user_gender"],
        "time_aggregation": "hour",
        "time_basis": "origin",
        "min_trips_per_flow": 1,
    }

    metadata = {
        "dataset_id": dataset_id,
        "is_validated": bool(validated),
        "events": [
            {
                "op": "build_flows",
                "ts_utc": "2026-04-01T12:00:00Z",
                "parameters": {
                    "h3_resolution": 8,
                    "group_by": ["mode", "day_type", "user_gender"],
                    "time_aggregation": "hour",
                    "time_basis": "origin",
                    "min_trips_per_flow": 1,
                },
                "summary": {
                    "n_flows": int(len(flows_df)),
                    "n_trips_in": int(len(flows_df) * 4),
                    "n_trips_aggregated": int(len(flows_df) * 4),
                    "n_trips_dropped": 0,
                    "n_flow_to_trips_rows": (
                        int(len(flow_to_trips_df))
                        if flow_to_trips_df is not None
                        else None
                    ),
                },
                "issues_summary": {
                    "counts": {"info": 0, "warning": 0, "error": 0},
                    "top_codes": [],
                },
            }
        ],
        "notes": {"fixture": "integration_rich_flowdataset"},
    }

    provenance = {
        "derived_from": [
            {
                "source_type": "trips",
                "dataset_id": "trip-dset-origin-001",
                "schema_version": "1.1",
            }
        ],
        "prior_events_summary": {"n_events": 3},
    }

    return FlowDataset(
        flows=flows_df,
        flow_to_trips=flow_to_trips_df,
        aggregation_spec=aggregation_spec,
        source_trips={"debug": "in_memory_only"},
        metadata=metadata,
        provenance=provenance,
    )

### 0.5 Inicialización de entorno y fixtures reutilizables

In [5]:
reset_integration_root()

flowdataset_small = make_rich_flowdataset(
    repeat_blocks=1,
    with_trip_links=False,
    validated=False,
    dataset_id="flow-dset-small-001",
)

flowdataset_with_trip_links = make_rich_flowdataset(
    repeat_blocks=1,
    with_trip_links=True,
    validated=False,
    dataset_id="flow-dset-links-001",
)

print("INTEGRATION_ROOT =", INTEGRATION_ROOT.resolve())
print("ARTIFACTS_ROOT =", ARTIFACTS_ROOT.resolve())
print("flowdataset_small.flows.shape =", flowdataset_small.flows.shape)
print("flowdataset_with_trip_links.flows.shape =", flowdataset_with_trip_links.flows.shape)
print(
    "flowdataset_with_trip_links.flow_to_trips.shape =",
    flowdataset_with_trip_links.flow_to_trips.shape,
)

display(flowdataset_small.flows.head(3))
show_ok("Sección 0 - setup y fixtures ricas para integración de OP-10")

INTEGRATION_ROOT = C:\projects\pylondrina\notebooks\testing\io_flows\tmp_integration_write_flows
ARTIFACTS_ROOT = C:\projects\pylondrina\notebooks\testing\io_flows\tmp_integration_write_flows\artifacts
flowdataset_small.flows.shape = (240, 15)
flowdataset_with_trip_links.flows.shape = (240, 15)
flowdataset_with_trip_links.flow_to_trips.shape = (720, 2)


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,purpose,day_type,user_gender,income_quintile,time_period,window_start_utc,window_end_utc,avg_trip_weight,segment_label
0,f_00_00000,8828308281fffff,8828308291fffff,5,5.25,bus,work,weekday,female,1,morning_peak,2026-04-01 06:00:00+00:00,2026-04-01 07:00:00+00:00,0.80,bus|weekday|female
1,f_00_00001,8828308281fffff,8828308291fffff,6,6.30,bus,education,weekday,male,3,midday,2026-04-01 07:00:00+00:00,2026-04-01 08:00:00+00:00,1.01,bus|weekday|male
2,f_00_00002,8828308281fffff,8828308291fffff,7,7.35,bus,shopping,weekend,female,5,afternoon_peak,2026-04-01 08:00:00+00:00,2026-04-01 09:00:00+00:00,1.22,bus|weekend|female


OK - Sección 0 - setup y fixtures ricas para integración de OP-10


## Bloque 1 - write feliz con FlowDataset rica

Qué prueba:

- camino principal correcto de `write_flows`;
- persistencia formal en Parquet con normalización automática a `.golondrina`;
- creación de archivo tabular y sidecar;
- coherencia de `summary` y `parameters`;
- registro de evento `write_flows`;
- sidecar formal consistente;
- preservación de `flows.flows` y de `source_trips` en memoria;
- no serialización de `source_trips` dentro del artefacto.

In [6]:
case_dir = make_case_dir("case_01_write_happy")
artifact_path = case_dir / "flows_write_happy"

flows = copy.deepcopy(flowdataset_small)
flows_before = flows.flows.copy(deep=True)
source_trips_before = copy.deepcopy(flows.source_trips)

report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=True,
        write_flow_to_trips=False,
    ),
)

effective_root = artifact_path.with_name(
    artifact_path.name + ".golondrina"
)
sidecar_path = effective_root / "flows.metadata.json"
sidecar = read_json(sidecar_path)

assert report.ok is True

# Layout final
assert effective_root.exists()
assert (effective_root / "flows.parquet").exists()
assert sidecar_path.exists()
assert not (effective_root / "flow_to_trips.parquet").exists()

# Summary
assert report.summary["n_flows"] == len(flows.flows)
assert report.summary["n_flow_to_trips"] is None
assert report.summary["dataset_id"] == flows.metadata["dataset_id"]
assert report.summary["artifact_id"] == flows.metadata["artifact_id"]
assert report.summary["path"] == str(effective_root)
assert set(report.summary["files_written"]) == {
    "flows.parquet",
    "flows.metadata.json",
}

# Parameters efectivos
assert report.parameters["path"] == str(effective_root)
assert report.parameters["mode"] == "error_if_exists"
assert report.parameters["storage_format"] == "parquet"
assert report.parameters["parquet_compression"] == "snappy"
assert report.parameters["normalize_artifact_dir"] is True
assert report.parameters["write_flow_to_trips"] is False

# Side effects en memoria
assert "artifact_id" in flows.metadata
assert flows.source_trips == source_trips_before
pd.testing.assert_frame_equal(flows.flows, flows_before)

# Sidecar formal
assert sidecar["dataset_type"] == "flows"
assert sidecar["format"] == "golondrina"
assert sidecar["layout_version"] == "1.1"
assert sidecar["storage"]["format"] == "parquet"
assert sidecar["files"]["data"] == "flows.parquet"
assert sidecar["files"]["metadata"] == "flows.metadata.json"
assert sidecar["files"]["flow_to_trips"] is None
assert sidecar["dataset_id"] == flows.metadata["dataset_id"]
assert sidecar["artifact_id"] == flows.metadata["artifact_id"]

# source_trips es referencia viva y no debe persistirse
assert "source_trips" not in sidecar
assert "source_trips" not in sidecar["metadata"]

# Evento write alineado con el reporte
event = flows.metadata["events"][-1]
assert event["op"] == "write_flows"
assert event["parameters"] == report.parameters
assert event["summary"] == report.summary
assert "issues_summary" in event

display(report)
show_ok("Bloque 1 - write feliz con FlowDataset rica")

OperationReport(ok=True, issues=[], summary={'n_flows': 240, 'n_flow_to_trips': None, 'files_written': ['flows.parquet', 'flows.metadata.json'], 'dataset_id': 'flow-dset-small-001', 'artifact_id': 'art_1367ffbb-dcfc-44e8-875b-80d8b1af7eaa', 'path': 'tmp_integration_write_flows\\artifacts\\case_01_write_happy\\flows_write_happy.golondrina'}, parameters={'path': 'tmp_integration_write_flows\\artifacts\\case_01_write_happy\\flows_write_happy.golondrina', 'mode': 'error_if_exists', 'storage_format': 'parquet', 'parquet_compression': 'snappy', 'feather_compression': 'lz4', 'normalize_artifact_dir': True, 'write_flow_to_trips': False})

OK - Bloque 1 - write feliz con FlowDataset rica


## Bloque 2 - write con auxiliar y verificación observable de dictionary encoding

Qué prueba:

- persistencia correcta de `flow_to_trips`;
- escritura formal en Parquet sin normalización del directorio;
- inclusión del auxiliar en `files_written`;
- conteo correcto de filas auxiliares en `summary`;
- dictionary encoding observable en campos categóricos de segmentación:
  `mode`, `day_type` y `user_gender`;
- comparación visible de tamaño contra una escritura manual sin dictionary encoding.

Este bloque se conserva como integración de OP-10 porque combina:
dataset rico + persistencia completa + auxiliar + inspección física del archivo generado.

In [7]:
case_dir = make_case_dir("case_02_write_with_aux_and_dictionary")
artifact_path = case_dir / "flows_with_aux"

flows = make_rich_flowdataset(
    repeat_blocks=20,
    with_trip_links=True,
    validated=False,
    dataset_id="flow-dset-dict-001",
)

report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

good_parquet = artifact_path / "flows.parquet"
aux_parquet = artifact_path / "flow_to_trips.parquet"

assert report.ok is True
assert good_parquet.exists()
assert aux_parquet.exists()

assert report.summary["n_flow_to_trips"] == len(flows.flow_to_trips)
assert "flow_to_trips.parquet" in report.summary["files_written"]

# Dictionary encoding observable en columnas categóricas de group_by
assert parquet_has_dictionary_encoding(good_parquet, "mode") is True
assert parquet_has_dictionary_encoding(good_parquet, "day_type") is True
assert parquet_has_dictionary_encoding(good_parquet, "user_gender") is True

# Comparación visible contra escritura manual sin dictionary encoding
bad_path = case_dir / "flows_no_dictionary_manual.parquet"

table_no_dict = pa.Table.from_pandas(
    flows.flows.copy(deep=True),
    preserve_index=False,
)

pq.write_table(
    table_no_dict,
    bad_path,
    compression="snappy",
    use_dictionary=False,
)

size_good = good_parquet.stat().st_size
size_bad = bad_path.stat().st_size

print("size_good =", size_good)
print("size_bad  =", size_bad)
print("ratio_bad_over_good =", round(size_bad / size_good, 3))

# No lo hacemos ultra rígido, pero al menos no debería empeorar.
assert size_good <= size_bad

display(report.summary)
show_ok("Bloque 2 - write con auxiliar y dictionary encoding observable")

size_good = 43223
size_bad  = 68706
ratio_bad_over_good = 1.59


{'n_flows': 4800,
 'n_flow_to_trips': 14400,
 'files_written': ['flows.parquet',
  'flows.metadata.json',
  'flow_to_trips.parquet'],
 'dataset_id': 'flow-dset-dict-001',
 'artifact_id': 'art_b768bcc1-76a2-4acd-bfdb-a5fb8c24c6c0',
 'path': 'tmp_integration_write_flows\\artifacts\\case_02_write_with_aux_and_dictionary\\flows_with_aux'}

OK - Bloque 2 - write con auxiliar y dictionary encoding observable


## Bloque 3 - write fatal por colisión de destino con `mode="error_if_exists"`

Qué prueba:

- primera escritura exitosa de un bundle formal;
- segunda escritura al mismo destino bajo `mode="error_if_exists"` falla;
- el error expuesto corresponde a una falla operacional de exportación/persistencia,
  no a un resultado silenciosamente sobrescrito.

In [9]:
case_dir = make_case_dir("case_03_write_collision")
artifact_path = case_dir / "flows_collision"

flows = copy.deepcopy(flowdataset_small)

# Primera escritura exitosa
report_ok = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)

assert report_ok.ok is True
assert (artifact_path / "flows.parquet").exists()

# Segunda escritura al mismo destino debe fallar
raised = None

try:
    write_flows(
        copy.deepcopy(flowdataset_small),
        artifact_path,
        options=WriteFlowsOptions(
            mode="error_if_exists",
            storage_format="parquet",
            normalize_artifact_dir=False,
            write_flow_to_trips=False,
        ),
    )
except Exception as exc:
    raised = exc

assert raised is not None
assert isinstance(raised, ExportError)

display(raised)
show_ok("Bloque 3 - write fatal por colisión de destino")

ExportError(message='Falló la promoción del staging al destino final del bundle .golondrina.', code='WRITE_FLOWS.IO.COMMIT_FAILED', details={'path': 'tmp_integration_write_flows\\artifacts\\case_03_write_collision\\flows_collision', 'artifact': 'flows.parquet, flows.metadata.json', 'mode': 'error_if_exists', 'reason': 'ExportError: El bundle destino ya existe y mode=\'error_if_exists\'; write_flows abortado.\ncode: WRITE_FLOWS.LAYOUT.BUNDLE_EXISTS\ndetails:\n{\'path\': \'tmp_integration_write_flows\\\\artifacts\\\\case_03_write_collision\\\\flows_collision\',\n \'mode\': \'error_if_exists\',\n \'artifact\': \'flows_collision\',\n \'reason\': \'bundle_already_exists\',\n \'action\': \'abort\'}\nissue: Issue(level=\'error\', code=\'WRITE_FLOWS.LAYOUT.BUNDLE_EXISTS\', message="El bundle destino ya existe y mode=\'error_if_exists\'; write_flows abortado.", field=None, source_field=None, row_count=None, details={\'path\': \'tmp_integration_write_flows\\\\artifacts\\\\case_03_write_collision

OK - Bloque 3 - write fatal por colisión de destino


## Bloque 4 - write feliz con backend default Feather

Qué prueba:

- que `write_flows` usa Feather por defecto cuando no se indica `storage_format`;
- que escribe `flows.feather`;
- que no escribe `flows.parquet`;
- que el sidecar queda consistente con backend Feather;
- que `parameters`, `summary` y evento quedan alineados;
- que la tabla principal y `source_trips` no se mutan en memoria.

En este caso se fija explícitamente `feather_compression="uncompressed"`,
por lo que esa decisión debe aparecer tanto en `report.parameters`
como en el sidecar persistido.

In [10]:
case_dir = make_case_dir("case_04_write_default_feather")
artifact_path = case_dir / "flows_write_default_feather"

flows = copy.deepcopy(flowdataset_small)
flows_before = flows.flows.copy(deep=True)
source_trips_before = copy.deepcopy(flows.source_trips)

report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
        feather_compression="uncompressed",
    ),
)

sidecar_path = artifact_path / "flows.metadata.json"
sidecar = read_json(sidecar_path)

assert report.ok is True

# Layout físico esperado
assert artifact_path.exists()
assert (artifact_path / "flows.feather").exists()
assert not (artifact_path / "flows.parquet").exists()
assert sidecar_path.exists()
assert not (artifact_path / "flow_to_trips.feather").exists()

# Parameters
assert report.parameters["storage_format"] == "feather"
assert report.parameters["feather_compression"] == "uncompressed"
assert report.parameters["write_flow_to_trips"] is False

# Summary
assert report.summary["n_flows"] == len(flows.flows)
assert report.summary["n_flow_to_trips"] is None
assert report.summary["dataset_id"] == flows.metadata["dataset_id"]
assert report.summary["artifact_id"] == flows.metadata["artifact_id"]
assert report.summary["path"] == str(artifact_path)
assert set(report.summary["files_written"]) == {
    "flows.feather",
    "flows.metadata.json",
}

# No mutación de estado vivo relevante
pd.testing.assert_frame_equal(flows.flows, flows_before)
assert flows.source_trips == source_trips_before

# Sidecar consistente con backend Feather
assert sidecar["dataset_type"] == "flows"
assert sidecar["format"] == "golondrina"
assert sidecar["layout_version"] == "1.1"
assert sidecar["storage"]["format"] == "feather"
assert sidecar["storage"]["options"]["compression"] == "uncompressed"
assert sidecar["storage"]["options"]["version"] == 2
assert sidecar["files"]["data"] == "flows.feather"
assert sidecar["files"]["metadata"] == "flows.metadata.json"
assert sidecar["files"]["flow_to_trips"] is None
assert sidecar["dataset_id"] == flows.metadata["dataset_id"]
assert sidecar["artifact_id"] == flows.metadata["artifact_id"]

# Evento write alineado
event = flows.metadata["events"][-1]
assert event["op"] == "write_flows"
assert event["parameters"] == report.parameters
assert event["summary"] == report.summary
assert "issues_summary" in event

display(report)
display(sidecar)
show_ok("Bloque 4 - write feliz con backend default Feather")

OperationReport(ok=True, issues=[], summary={'n_flows': 240, 'n_flow_to_trips': None, 'files_written': ['flows.feather', 'flows.metadata.json'], 'dataset_id': 'flow-dset-small-001', 'artifact_id': 'art_56e97997-3f3f-401f-b492-6e1dc9514c46', 'path': 'tmp_integration_write_flows\\artifacts\\case_04_write_default_feather\\flows_write_default_feather'}, parameters={'path': 'tmp_integration_write_flows\\artifacts\\case_04_write_default_feather\\flows_write_default_feather', 'mode': 'error_if_exists', 'storage_format': 'feather', 'parquet_compression': 'snappy', 'feather_compression': 'uncompressed', 'normalize_artifact_dir': False, 'write_flow_to_trips': False})

{'dataset_type': 'flows',
 'format': 'golondrina',
 'layout_version': '1.1',
 'storage': {'format': 'feather',
  'options': {'compression': 'uncompressed', 'version': 2}},
 'dataset_id': 'flow-dset-small-001',
 'artifact_id': 'art_56e97997-3f3f-401f-b492-6e1dc9514c46',
 'files': {'data': 'flows.feather',
  'metadata': 'flows.metadata.json',
  'flow_to_trips': None},
 'aggregation_spec': {'h3_resolution': 8,
  'group_by': ['mode', 'day_type', 'user_gender'],
  'time_aggregation': 'hour',
  'time_basis': 'origin',
  'min_trips_per_flow': 1},
 'provenance': {'derived_from': [{'source_type': 'trips',
    'dataset_id': 'trip-dset-origin-001',
    'schema_version': '1.1'}],
  'prior_events_summary': {'n_events': 3}},
 'metadata': {'dataset_id': 'flow-dset-small-001',
  'is_validated': False,
  'events': [{'op': 'build_flows',
    'ts_utc': '2026-04-01T12:00:00Z',
    'parameters': {'h3_resolution': 8,
     'group_by': ['mode', 'day_type', 'user_gender'],
     'time_aggregation': 'hour',
    

OK - Bloque 4 - write feliz con backend default Feather


## Bloque 5 - write explícito en Parquet sigue funcionando

Qué prueba:

- que la incorporación de Feather como backend por defecto
  no rompió la ruta explícita de escritura en Parquet;
- que el archivo generado es `flows.parquet`;
- que no aparece `flows.feather`;
- que `parameters`, `summary`, sidecar y evento
  mantienen consistencia con el backend Parquet.

Este bloque funciona como integración de regresión sobre el soporte Parquet.

In [11]:
case_dir = make_case_dir("case_05_write_explicit_parquet_still_works")
artifact_path = case_dir / "flows_write_explicit_parquet"

flows = copy.deepcopy(flowdataset_small)
flows_before = flows.flows.copy(deep=True)

report = write_flows(
    flows,
    artifact_path,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)

sidecar_path = artifact_path / "flows.metadata.json"
sidecar = read_json(sidecar_path)

assert report.ok is True

# Layout físico esperado
assert artifact_path.exists()
assert (artifact_path / "flows.parquet").exists()
assert not (artifact_path / "flows.feather").exists()
assert sidecar_path.exists()

# Parameters
assert report.parameters["storage_format"] == "parquet"
assert report.parameters["parquet_compression"] == "snappy"
assert report.parameters["write_flow_to_trips"] is False

# Summary
assert report.summary["n_flows"] == len(flows.flows)
assert report.summary["n_flow_to_trips"] is None
assert set(report.summary["files_written"]) == {
    "flows.parquet",
    "flows.metadata.json",
}

# No mutación de tabla principal
pd.testing.assert_frame_equal(flows.flows, flows_before)

# Sidecar consistente
assert sidecar["storage"]["format"] == "parquet"
assert sidecar["storage"]["options"]["compression"] == "snappy"
assert sidecar["files"]["data"] == "flows.parquet"
assert sidecar["files"]["metadata"] == "flows.metadata.json"
assert sidecar["files"]["flow_to_trips"] is None

# Evento write alineado
event = flows.metadata["events"][-1]
assert event["op"] == "write_flows"
assert event["parameters"] == report.parameters
assert event["summary"] == report.summary
assert "issues_summary" in event

display(report)
display(sidecar)
show_ok("Bloque 5 - write explícito en Parquet sigue funcionando")

OperationReport(ok=True, issues=[], summary={'n_flows': 240, 'n_flow_to_trips': None, 'files_written': ['flows.parquet', 'flows.metadata.json'], 'dataset_id': 'flow-dset-small-001', 'artifact_id': 'art_d5d24b31-051c-4f44-967f-7a4fca18c890', 'path': 'tmp_integration_write_flows\\artifacts\\case_05_write_explicit_parquet_still_works\\flows_write_explicit_parquet'}, parameters={'path': 'tmp_integration_write_flows\\artifacts\\case_05_write_explicit_parquet_still_works\\flows_write_explicit_parquet', 'mode': 'error_if_exists', 'storage_format': 'parquet', 'parquet_compression': 'snappy', 'feather_compression': 'lz4', 'normalize_artifact_dir': False, 'write_flow_to_trips': False})

{'dataset_type': 'flows',
 'format': 'golondrina',
 'layout_version': '1.1',
 'storage': {'format': 'parquet', 'options': {'compression': 'snappy'}},
 'dataset_id': 'flow-dset-small-001',
 'artifact_id': 'art_d5d24b31-051c-4f44-967f-7a4fca18c890',
 'files': {'data': 'flows.parquet',
  'metadata': 'flows.metadata.json',
  'flow_to_trips': None},
 'aggregation_spec': {'h3_resolution': 8,
  'group_by': ['mode', 'day_type', 'user_gender'],
  'time_aggregation': 'hour',
  'time_basis': 'origin',
  'min_trips_per_flow': 1},
 'provenance': {'derived_from': [{'source_type': 'trips',
    'dataset_id': 'trip-dset-origin-001',
    'schema_version': '1.1'}],
  'prior_events_summary': {'n_events': 3}},
 'metadata': {'dataset_id': 'flow-dset-small-001',
  'is_validated': False,
  'events': [{'op': 'build_flows',
    'ts_utc': '2026-04-01T12:00:00Z',
    'parameters': {'h3_resolution': 8,
     'group_by': ['mode', 'day_type', 'user_gender'],
     'time_aggregation': 'hour',
     'time_basis': 'origin

OK - Bloque 5 - write explícito en Parquet sigue funcionando
